# Práctica 2 · Paso 2 — Vulnerabilidad y riesgo de los cantones prioritarios

**Cambio Climático · Ingeniería en Riesgos de Desastres · ESPAM MFL** · Material CC-IRD-P2-02 · Versión 1.0 · noviembre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-2/RD2_02_vulnerabilidad.ipynb)

**Objetivo.** Calificar la vulnerabilidad de los cantones más expuestos de su provincia (los que salieron en `RD2_01`) y calcular su **índice de riesgo** frente a cada amenaza del capítulo 1.

**Producto.** Planilla de vulnerabilidad justificada con datos y fuentes, tabla y mapa de riesgo por cantón y amenaza, y el cantón y la amenaza críticos que el paso 3 (`RD2_03`) usa para proponer medidas.

**Marco (IPCC AR6, GTII).** Riesgo = amenaza × exposición × vulnerabilidad. La vulnerabilidad junta dos cosas: la **sensibilidad** (cuánto daño sufre lo expuesto) y la **capacidad de adaptación** (cuánto puede prevenir, responder y recuperarse). En este cuaderno:

| Componente | De dónde sale | Escala |
|---|---|---|
| A · amenaza | Su capítulo 1 (`RD1_01`, `RD1_02`) y los eventos de la provincia | 1–5, una por amenaza, para toda la provincia |
| E · exposición | `RD2_01` (la matriz corregida por el equipo) | 1–5 por cantón y amenaza |
| S · sensibilidad | Planilla del equipo: 5 indicadores con dato y fuente | 1–5 por cantón (5 = muy sensible) |
| CA · capacidad de adaptación | Planilla del equipo: 4 indicadores con dato y fuente | 1–5 por cantón (**5 = muy buena**) |
| V · vulnerabilidad | V = (S + (6 − CA)) / 2 | 1–5 |
| R · riesgo | R = (A · E · V)^(1/3), media geométrica | 1–5: muy bajo < 1,8 ≤ bajo < 2,6 ≤ medio < 3,4 ≤ alto < 4,2 ≤ muy alto |

**Requisito.** Haber ejecutado `RD2_01_exposicion` con la misma provincia: este cuaderno lee sus productos de `Mi unidad/CC_Riesgos/practica-2/<provincia>/`.

**Guía para el estudiante.** Paso a paso, ejemplos y cómo llevar cada salida al capítulo 2: [`guia_RD2_02.md`](https://github.com/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-2/guia_RD2_02.md).

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Complete la celda **1**, ejecute todo en orden y, en la sección 6, llene la planilla de vulnerabilidad en Google Sheets. Después vuelva a ejecutar desde la sección 7. Tiempo estimado: 1 hora en clase y 2 horas sin docente.

**Con energía o internet intermitentes.** La planilla se puede llenar sin conexión (en Excel o en papel, con el mismo orden de filas) y pasarse a Drive después; el cuaderno solo lee archivos de su Drive y tarda menos de 2 minutos.

## 1. Datos del equipo

In [ ]:
# @title Equipo y provincia (la misma de RD2_01)
ESTUDIANTES = "Nombre Apellido · Nombre Apellido"  # @param {type:"string"}
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
print(f"Equipo: {ESTUDIANTES} · provincia de {PROVINCIA}")

## 2. Entorno de trabajo y productos de RD2_01

In [ ]:
import os, sys, pathlib, subprocess, datetime, json, re, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, riesgo as RG, exposicion as EX, vulnerabilidad as VU

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(os.environ.get("CC_BASE", "."))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, geopandas as gpd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
SP = limites.slug(PROVINCIA)
RES = BASE / "CC_Riesgos" / "practica-2" / SP
NECESARIOS = [RES / "RD2_01_metadatos.json", RES / f"RD2_01_puntajes_exposicion_{SP}.csv", RES / f"RD2_01_cantones_prioritarios_{SP}.csv"]
faltan = [p.name for p in NECESARIOS if not p.exists()]
if faltan:
    raise FileNotFoundError(f"Faltan productos de RD2_01 en {RES}: {faltan}.\nEjecute RD2_01_exposicion con la provincia {PROVINCIA} (la misma de la celda 1).")
META1 = json.loads(NECESARIOS[0].read_text(encoding="utf-8"))
AMENAZAS = META1["amenazas"]
PUNT = pd.read_csv(NECESARIOS[1], dtype={"codigo": str})
PRIOR = pd.read_csv(NECESARIOS[2])
CANTONES = list(dict.fromkeys(PRIOR["canton"]))
P1 = pathlib.Path(META1.get("capitulo_1", ""))
print(f"Productos de RD2_01 del {META1.get('fecha', '?')[:10]} en {RES}")
print(f"Amenazas priorizadas: {', '.join(RG.AMENAZA_CORTA[a] for a in AMENAZAS)}")
print(f"Cantones prioritarios ({len(CANTONES)}): {', '.join(CANTONES)}")

## 3. Lo que trae del paso 1

Revise que la exposición y el inventario estén completos: la vulnerabilidad se califica **solo para los cantones prioritarios** de `RD2_01`, y el inventario de elementos expuestos es la base para saber qué es sensible en cada cantón (el barrio de caña junto al río, la escuela sin drenaje).

In [ ]:
TAB_E = PUNT.set_index("canton").loc[CANTONES, AMENAZAS].rename(columns=RG.AMENAZA_CORTA)
print("Exposición (1–5) de los cantones prioritarios, de RD2_01:"); display(TAB_E)
RUTA_INV = RES / f"RD2_01_inventario_expuestos_{SP}.xlsx"
INV_OK = {}
if RUTA_INV.exists():
    INV = pd.read_excel(RUTA_INV, sheet_name="inventario").fillna("")
    llenas = INV[(INV["nombre_o_lugar"].astype(str).str.strip() != "") & (INV["fuente"].astype(str).str.strip() != "")]
    INV_OK = llenas.groupby("canton").size().reindex(CANTONES, fill_value=0)
    print("\nElementos del inventario con nombre y fuente, por cantón:"); display(INV_OK.rename("elementos").to_frame())
    pocos = INV_OK[INV_OK < 3]
    if len(pocos):
        print(f"⚠ Con menos de 3 elementos concretos: {', '.join(pocos.index)}. Complete el inventario de RD2_01 (sección 9): lo necesita para justificar la sensibilidad.")
else:
    print(f"⚠ No encontré {RUTA_INV.name}: ejecute la sección 9 de RD2_01 y complete la planilla.")

## 4. La amenaza de la provincia (A)

La amenaza se califica **una vez para la provincia**, de 1 a 5, con la evidencia de su capítulo 1. El cuaderno le propone una **sugerencia** con dos datos: cuántos desastres registró DesInventar en la provincia (1970–2007) y, si encuentra los productos de la Práctica 1, cuánto aumenta la lluvia de la temporada en los años de El Niño o cuánto suben los días cálidos. La sugerencia es un punto de partida: la calificación final es del equipo y se justifica.

| Valor | Descriptor |
|---|---|
| 1 | No hay registros ni evidencia física |
| 2 | Evidencia débil o eventos muy raros |
| 3 | Eventos ocasionales; evidencia en el territorio |
| 4 | Eventos frecuentes o intensos; con El Niño se agravan claramente |
| 5 | Eventos recurrentes y severos; daños documentados repetidos |

In [ ]:
FILA_P, _ = limites.provincia(PROVINCIA)
EV = EX.eventos_por_canton(FILA_P["codigo"])
N_DES = EV[EV.fuente == "DesInventar"].groupby("amenaza")["eventos"].sum() if len(EV) else pd.Series(dtype=int)
N_911 = EV[EV.fuente == "ECU 911"].groupby("amenaza")["eventos"].sum() if len(EV) else pd.Series(dtype=int)

FACTOR_NINO, TX90_DEC = None, None
try:   # lluvia de la temporada en años de El Niño frente a los demás (RD1_01)
    t1 = sorted(P1.glob("RD1_01_temporada_enso_*.csv"))
    m1 = json.loads((P1 / "RD1_01_metadatos.json").read_text(encoding="utf-8"))
    if t1:
        TT = pd.read_csv(t1[0], index_col=0)
        col = [c for c in TT.columns if c.endswith("_mm")][0]
        nino = TT.index.isin(m1["anios_nino_temporada"])
        FACTOR_NINO = round(float(TT.loc[nino, col].mean() / TT.loc[~nino, col].mean()), 2)
except Exception:
    pass
try:   # tendencia de los días cálidos (RD1_02)
    t2 = sorted(P1.glob("RD1_02_tendencias_*.csv"))
    if t2:
        TD = pd.read_csv(t2[0], index_col=[0, 1])
        fila = [i for i in TD.index if i[1] == "TX90p"]
        if fila:
            TX90_DEC = round(float(TD.loc[fila[0], "sen_por_decada"]), 2)
except Exception:
    pass

SUG, BASE_SUG = {}, {}
for a in RG.AMENAZAS:
    s_ev = VU.sugerencia_amenaza(int(N_DES.get(a, 0)))
    s, b = s_ev, f"{int(N_DES.get(a, 0))} desastres en DesInventar"
    if a in ("inundacion", "movimiento_masa") and FACTOR_NINO:
        s_ll = RG.sugerencia_lluvia(FACTOR_NINO)
        if s_ll > s: s, b = s_ll, f"lluvia × {FACTOR_NINO} en años de El Niño (RD1_01)"
    if a == "calor":
        s, b = (RG.sugerencia_calor(TX90_DEC), f"TX90p {TX90_DEC:+.1f} %/década (RD1_02)") if TX90_DEC is not None else (0, "sin dato de RD1_02: califique a mano")
    SUG[a], BASE_SUG[a] = s, b
TAB_A = pd.DataFrame({"amenaza": [RG.AMENAZA_CORTA[a] for a in AMENAZAS],
                      "desastres DesInventar 1970–2007": [int(N_DES.get(a, 0)) for a in AMENAZAS],
                      "días ECU 911 desde 2021": [int(N_911.get(a, 0)) for a in AMENAZAS],
                      "sugerencia": [SUG[a] for a in AMENAZAS], "por qué": [BASE_SUG[a] for a in AMENAZAS],
                      "justificación de RD2_01": [META1["justificacion"].get(a, "") for a in AMENAZAS]})
display(TAB_A)
print("Factor de El Niño (RD1_01):", FACTOR_NINO if FACTOR_NINO else "no encontrado", "· TX90p (RD1_02):", TX90_DEC if TX90_DEC is not None else "no encontrado")

In [ ]:
# @title Calificación de la amenaza (0 = usar la sugerencia del cuaderno). Solo cuentan las amenazas priorizadas en RD2_01
A_INUNDACION = 0  # @param {type:"slider", min:0, max:5, step:1}
A_MOVIMIENTO_MASA = 0  # @param {type:"slider", min:0, max:5, step:1}
A_COSTERA = 0  # @param {type:"slider", min:0, max:5, step:1}
A_SEQUIA = 0  # @param {type:"slider", min:0, max:5, step:1}
A_CALOR = 0  # @param {type:"slider", min:0, max:5, step:1}
J_AMENAZA = ""  # @param {type:"string"}
# Escriba por qué cambió alguna sugerencia. Ejemplo: «Inundación 5: Rx1day de 25 años = ___ mm (RD1_02) y ___ desastres en DesInventar, muchos en El Niño 1997–98».

_sl = {"inundacion": A_INUNDACION, "movimiento_masa": A_MOVIMIENTO_MASA, "costera": A_COSTERA, "sequia": A_SEQUIA, "calor": A_CALOR}
A = {}
for a in AMENAZAS:
    v = _sl[a] or SUG[a]
    if not v:
        raise ValueError(f"No hay sugerencia para «{RG.AMENAZA_CORTA[a]}»: califíquela con el deslizador (1–5).")
    A[a] = int(v)
CAMBIADAS = [a for a in AMENAZAS if _sl[a] and _sl[a] != SUG[a]]
for a in AMENAZAS:
    print(f"· {RG.AMENAZA_CORTA[a]}: A = {A[a]}" + (f" (sugerencia {SUG[a]})" if a in CAMBIADAS else " (sugerencia del cuaderno)"))
if CAMBIADAS and not J_AMENAZA.strip():
    print("⚠ Cambió la sugerencia de " + ", ".join(RG.AMENAZA_CORTA[a] for a in CAMBIADAS) + ": escriba la razón en J_AMENAZA.")

## 5. Datos de apoyo por cantón

Estos indicadores salen de los datos del curso y **ayudan** a calificar; no reemplazan el dato del Censo o del PDOT que el equipo busca.

| Indicador | Para qué sirve | Cuidado |
|---|---|---|
| % población en zona baja o en ladera | Recordar dónde está la gente expuesta | Es exposición, no sensibilidad |
| Salud y emergencia por 10 000 hab | Capacidad: **acceso** a servicios | OpenStreetMap puede estar incompleto en cantones rurales |
| Vías principales por 100 km² | Capacidad: **acceso** y evacuación | No dice si la vía se corta en invierno |
| Afectados históricos por 1 000 hab | Cuánto daño hubo antes: indicio de sensibilidad | DesInventar llega a 2007; una cifra vacía es «no reportado» |
| Sugerencia para «acceso» | Posición del cantón en el país (quintil) | Úsela como punto de partida |

In [ ]:
PAIS = EX.de_pais()
T = EX.de_provincia(PROVINCIA, EV)
AP = VU.apoyo(T, EV, PAIS)
TAB_AP = AP.set_index("canton").loc[[c for c in CANTONES if c in set(AP.canton)], list(VU.NOMBRES_APOYO)].rename(columns=VU.NOMBRES_APOYO)
display(TAB_AP.round(1))

## 6. Planilla de vulnerabilidad (trabajo del equipo)

El cuaderno crea `RD2_02_vulnerabilidad_<provincia>.xlsx` en su carpeta de resultados, con **9 filas por cantón prioritario**: 5 de sensibilidad y 4 de capacidad de adaptación. Cada fila trae **cómo calificar** (la escala 1–5 con umbrales orientativos) y **dónde buscar** el dato.

**Qué llena el equipo en cada fila:**

| Columna | Qué escribir | Ejemplo |
|---|---|---|
| `dato_encontrado` | El número o el hecho | «38 % de viviendas sin alcantarillado» |
| `valor` | La calificación, de 1 a 5 | 3 |
| `fuente` | De dónde sale | «INEC, Censo 2022, resultados por cantón» |
| `justificacion` | Por qué ese valor, en una línea | «Entre 30 y 50 %: valor 3; en la parroquia urbana es menor» |

**Atención: en la capacidad, 5 es lo mejor** (alerta temprana operativa, PDOT con ordenanza, fondos). En la sensibilidad, 5 es lo peor.

Abra el archivo con Google Sheets, complételo y guárdelo **con el mismo nombre y en la misma carpeta** (en Sheets: **Archivo › Descargar › Microsoft Excel** y súbalo reemplazando el original, o trabaje directamente en Excel). Luego ejecute desde la sección 7. Si la planilla ya existe, el cuaderno **no la sobrescribe**: solo agrega filas si aparece un cantón nuevo.

In [ ]:
RUTA_V = RES / f"RD2_02_vulnerabilidad_{SP}.xlsx"
NUEVA = VU.plantilla(CANTONES, AP)
if RUTA_V.exists():
    PL = pd.read_excel(RUTA_V, sheet_name="calificacion", dtype=str).fillna("")
    extra = NUEVA[~NUEVA.canton.isin(set(PL.canton))]
    if len(extra):
        PL = pd.concat([PL, extra.astype(str)], ignore_index=True)
        print(f"Agregué las filas de {', '.join(extra.canton.unique())} a la planilla existente.")
    else:
        print(f"La planilla ya existe ({RUTA_V.name}): no la sobrescribo.")
else:
    PL = NUEVA
if not RUTA_V.exists() or len(PL) != len(pd.read_excel(RUTA_V, sheet_name="calificacion")):
    with pd.ExcelWriter(RUTA_V) as w:
        PL.to_excel(w, sheet_name="calificacion", index=False)
        pd.DataFrame({"paso": ["1", "2", "3", "4", "5", "6"],
                      "qué hacer": ["Reparta los cantones entre los dos integrantes; cada uno busca los datos de los suyos",
                                    "Lea «como_calificar» y «donde_buscar» de cada fila",
                                    "Busque el dato (Censo 2022, PDOT, SNGR) y escríbalo en «dato_encontrado» con su unidad",
                                    "Ponga el valor de 1 a 5 según la escala de «como_calificar» (en la capacidad, 5 es lo mejor)",
                                    "Escriba la fuente y una línea de justificación",
                                    "Guarde el archivo con el mismo nombre y ejecute el cuaderno desde la sección 7"]}
                     ).to_excel(w, sheet_name="instrucciones", index=False)
    print(f"Planilla guardada: {RUTA_V}")
display(PL[["canton", "componente", "indicador", "dato_de_apoyo", "valor"]].head(9))

## 7. Vulnerabilidad por cantón

El cuaderno lee la planilla, revisa que cada valor esté entre 1 y 5 y que tenga fuente y justificación, y calcula S, CA y V. **Si hay valores vacíos usa 3 y marca el resultado como PROVISIONAL**: sirve para ver el ejercicio en clase, pero no se entrega así.

In [ ]:
PL = pd.read_excel(RUTA_V, sheet_name="calificacion", dtype=str).fillna("")
CAL, AVISOS_V, PROVISIONAL = VU.leer_calificaciones(PL, CANTONES)
CAL = CAL[CAL.canton.isin(CANTONES)]
VUL = VU.vulnerabilidad(CAL)
TAB_V = VUL.assign(sensibilidad_critica=VUL.sensibilidad_critica.map(VU.SENSIBILIDAD),
                   capacidad_debil=VUL.capacidad_debil.map(VU.CAPACIDAD))
display(TAB_V)
print("\n".join("⚠ " + a for a in AVISOS_V) or "Planilla completa.")

In [ ]:
ETQ = " (PROVISIONAL)" if PROVISIONAL else ""
fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(VUL) + 3), 4.6))
x = np.arange(len(VUL)); w = 0.27
for k, (col, nom, colr) in enumerate((("S", "Sensibilidad", "#C2531F"), ("falta_de_capacidad", "Falta de capacidad (6 − CA)", "#E8A33D"),
                                      ("V", "Vulnerabilidad", "#7F3C8D"))):
    b = ax.bar(x + (k - 1) * w, VUL[col], w, label=nom, color=colr)
    ax.bar_label(b, fmt="%.1f", fontsize=9)
ax.set_xticks(x, VUL.index, rotation=20, ha="right"); ax.set_ylim(0, 5.6); ax.set_ylabel("1–5")
ax.legend(frameon=False, ncol=3, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{PROVINCIA}: vulnerabilidad de los cantones prioritarios{ETQ}", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(RES / f"RD2_02_vulnerabilidad_{SP}.png", dpi=200); plt.show()

## 8. Riesgo por cantón y amenaza

R = (A · E · V)^(1/3). Como es una media geométrica, **un componente bajo baja el riesgo aunque los otros sean altos**: un cantón muy vulnerable pero poco expuesto a una amenaza no queda en riesgo muy alto. La columna `pesa_mas` dice cuál de los tres componentes es el mayor.

In [ ]:
RIESGO = VU.riesgo_cantones(A, PUNT, VUL)
display(RIESGO[["canton", "amenaza", "A", "E", "V", "R", "clase", "pesa_mas"]])

M = RIESGO.pivot(index="canton", columns="amenaza", values="R").reindex([c for c in CANTONES if c in set(RIESGO.canton)])
K = RIESGO.pivot(index="canton", columns="amenaza", values="clase").reindex(M.index)
from matplotlib.colors import ListedColormap, BoundaryNorm
cm = ListedColormap([RG.COLOR_CLASE[c] for c in ("muy bajo", "bajo", "medio", "alto", "muy alto")])
nrm = BoundaryNorm([1, 1.8, 2.6, 3.4, 4.2, 5.01], cm.N)
fig, ax = plt.subplots(figsize=(1.9 * M.shape[1] + 3.5, 0.5 * len(M) + 1.8))
ax.imshow(M.values, cmap=cm, norm=nrm, aspect="auto")
ax.set_xticks(range(M.shape[1]), M.columns); ax.set_yticks(range(len(M)), M.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if np.isfinite(M.values[i, j]):
            ax.text(j, i, f"{M.values[i, j]:.1f}\n{K.values[i, j]}", ha="center", va="center", fontsize=9,
                    color="white" if M.values[i, j] >= 4.2 or M.values[i, j] < 1.8 else "black")
ax.set_title(f"{PROVINCIA}: riesgo por cantón y amenaza (1–5){ETQ}", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(RES / f"RD2_02_matriz_riesgo_{SP}.png", dpi=200); plt.show()

In [ ]:
CANT = limites.cargar("cantones")
RMAX = RIESGO.sort_values("R", ascending=False).drop_duplicates("canton")
RMAX = RMAX.merge(PUNT[["codigo", "canton"]], on="canton")
G = CANT.merge(T[["codigo"]], on="codigo").merge(RMAX[["codigo", "R", "clase", "amenaza"]], on="codigo", how="left")
fig, ax = plt.subplots(figsize=(7.5, 6.5))
G[G.R.isna()].plot(ax=ax, color="#EEEEEE", edgecolor="#999999", lw=0.4)
for c, g in G[G.R.notna()].groupby("clase"):
    g.plot(ax=ax, color=RG.COLOR_CLASE[c], edgecolor="#444444", lw=0.6)
for _, r in G[G.R.notna()].iterrows():
    p = r.geometry.representative_point()
    ax.annotate(f"{r.nombre_dpa.replace('Ð', 'Ñ').title()}\n{r.R:.1f} {r.amenaza.lower()}", (p.x, p.y), fontsize=7, ha="center")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=RG.COLOR_CLASE[c], label=c) for c in ("muy bajo", "bajo", "medio", "alto", "muy alto")] +
          [Patch(color="#EEEEEE", label="no evaluado")], loc="lower left", fontsize=8, title="Riesgo máximo")
ax.set_axis_off(); ax.set_title(f"{PROVINCIA}: riesgo máximo de los cantones prioritarios{ETQ}", fontweight="bold")
fig.tight_layout(); fig.savefig(RES / f"RD2_02_mapa_riesgo_{SP}.png", dpi=200); plt.show()

## 9. ¿Qué explica el riesgo? Frases de ejemplo para el capítulo

El cuaderno escribe una frase por cada combinación de mayor riesgo. **No las copie tal cual**: úselas como esqueleto y agregue el dato que encontró (el porcentaje del Censo, el nombre del barrio del inventario, el evento recordado).

In [ ]:
for _, r in RIESGO.head(3).iterrows():
    print("• " + VU.frase(r, VUL) + "\n")
CRIT = RIESGO.iloc[0]
CANTON_CRITICO, AMENAZA_CRITICA = CRIT.canton, CRIT.clave
print(f"Cantón y amenaza críticos (pasan al paso 3): {CANTON_CRITICO} · {CRIT.amenaza} (R = {CRIT.R:.1f}, {CRIT.clase})")
v = CAL[CAL.canton == CANTON_CRITICO]
FACTORES_CRITICOS = (v[(v.componente == "sensibilidad") & (v.valor >= 4)].clave.tolist() +
                     v[(v.componente == "capacidad") & (v.valor <= 2)].clave.tolist())
print("Indicadores críticos del cantón (sensibilidad ≥ 4 o capacidad ≤ 2):",
      ", ".join({**VU.SENSIBILIDAD, **VU.CAPACIDAD}[k] for k in FACTORES_CRITICOS) or "ninguno")
AVISOS = list(AVISOS_V)
if CAMBIADAS and not J_AMENAZA.strip(): AVISOS.append("Falta la justificación del cambio de la sugerencia de amenaza (J_AMENAZA).")
if len(set(RIESGO.clase)) == 1 and len(RIESGO) > 3: AVISOS.append("Todos los cantones quedaron en la misma clase de riesgo: revise si la planilla distingue los cantones.")
print("\n" + ("Sin observaciones." if not AVISOS else "Revise antes de entregar:\n- " + "\n- ".join(AVISOS)))

## 10. Guardar

In [ ]:
CAL.to_csv(RES / f"RD2_02_calificaciones_{SP}.csv", index=False, encoding="utf-8")
VUL.to_csv(RES / f"RD2_02_vulnerabilidad_{SP}.csv", encoding="utf-8")
RIESGO.to_csv(RES / f"RD2_02_riesgo_{SP}.csv", index=False, encoding="utf-8")
TAB_A.to_csv(RES / f"RD2_02_amenaza_{SP}.csv", index=False, encoding="utf-8")
meta = {"material": "CC-IRD-P2-02 v1.0", "equipo": ESTUDIANTES, "provincia": PROVINCIA, "amenazas": A,
        "sugerencias": {a: SUG[a] for a in AMENAZAS}, "base_sugerencias": {a: BASE_SUG[a] for a in AMENAZAS},
        "justificacion_amenaza": J_AMENAZA, "factor_nino": FACTOR_NINO, "tx90p_por_decada": TX90_DEC,
        "cantones": CANTONES, "canton_critico": CANTON_CRITICO, "amenaza_critica": AMENAZA_CRITICA,
        "amenazas_relevantes": RIESGO.loc[(RIESGO.canton == CANTON_CRITICO) & (RIESGO.R >= 2.6), "clave"].tolist(),
        "factores_criticos": FACTORES_CRITICOS, "provisional": PROVISIONAL, "avisos": AVISOS,
        "fecha": datetime.datetime.now().isoformat(timespec="seconds")}
(RES / "RD2_02_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Guardado en", RES)
for p in sorted(RES.glob("RD2_02_*")): print("  ", p.name)
if PROVISIONAL: print("\n⚠ Resultado PROVISIONAL: complete la planilla y vuelva a ejecutar desde la sección 7 antes de RD2_03.")

## 11. Preguntas para el capítulo 2 (sección 3, «Vulnerabilidad y riesgo»)

1. ¿Qué calificación de amenaza dio a cada amenaza y con qué evidencia del capítulo 1? Si cambió la sugerencia del cuaderno, ¿por qué?
2. ¿Qué cantón es el más vulnerable y qué indicadores lo explican? Cite el dato y la fuente de al menos dos.
3. ¿Qué cantón y qué amenaza tienen el mayor riesgo? ¿Cuál de los tres componentes (A, E o V) pesa más?
4. ¿Hay un cantón muy expuesto pero con riesgo menor que otro menos expuesto? Explíquelo con la vulnerabilidad.
5. ¿Qué parte del riesgo se puede reducir **sin cambiar la amenaza**? Señale los indicadores de sensibilidad y de capacidad que una medida podría mejorar: son el punto de partida del paso 3.
6. ¿Qué limitaciones tiene el índice? (la amenaza es de toda la provincia y no del cantón; las calificaciones dependen del dato que encontró; el promedio da el mismo peso a todos los indicadores).

**Para el informe:** la fórmula del índice y la escala de clases, la planilla completa en un anexo (o su tabla resumen), las fuentes de cada indicador y la fecha de consulta.

**Siguiente:** `RD2_03_matriz_medidas` usa el cantón crítico, sus amenazas de riesgo medio o mayor y sus indicadores críticos.

**Citas.** IPCC (2022). *Climate Change 2022: Impacts, Adaptation and Vulnerability*. Contribución del Grupo de Trabajo II al AR6, Anexo II: Glosario · GIZ y EURAC (2017). *Risk Supplement to the Vulnerability Sourcebook*. GIZ · Marin-Ferrer, M., Vernaccini, L. y Poljansek, K. (2017). *INFORM Index for Risk Management: Concept and Methodology*. JRC, EUR 28655 · INEC (2023). *Censo Ecuador 2022* · UNDRR (2015). *Marco de Sendai para la Reducción del Riesgo de Desastres 2015–2030*.